In [8]:
import os
import time
import joblib
import logging
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import MaxAbsScaler
from sklearn.cluster import KMeans

logging.basicConfig(level=logging.INFO)

In [9]:
def load_data(input_path):
    logging.info("Loading data...")
    df = pd.read_csv(os.path.join(input_path, 'telephish.csv'),
                     low_memory=False)
    return df

In [10]:
def preprocess_data(df):
    logging.info("Preprocessing data...")

    # Ensure boolean features are integers
    bool_features = ['is_forwarded', 'is_bot', 'has_media', 'has_text',
                     'is_reply', 'language_match', 'username_matches_language']
    for feature in bool_features:
        df[feature] = df[feature].astype(int)

    # Convert message time to categorical parts of the day
    df['time_of_day'] = df.apply(categorize_time, axis=1)
    df = pd.get_dummies(df, columns=['time_of_day'])

    return df

In [11]:
def categorize_time(row):
    """Categorize the time of day based on the message timestamp."""
    hour = pd.to_datetime(row['time']).hour
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Night'
    else:
        return 'Midnight'

In [12]:
def prepare_features(df, features):
    logging.info("Preparing features and target variables...")

    initial_row_count = len(df)
    df_dedup = df.drop_duplicates(subset=features)
    dedup_row_count = len(df_dedup)
    rows_dropped = initial_row_count - dedup_row_count
    logging.info(f"Number of duplicate rows dropped: {rows_dropped}")

    X = df_dedup[features]
    y = df_dedup['is_phishing']
    sample_ids = df_dedup["sample_ids"]
    category = df_dedup["category"]

    missing_in_X = X.isnull().sum()
    missing_columns_X = missing_in_X[missing_in_X > 0].index.tolist()
    if missing_columns_X:
        logging.warning(f"Columns with missing values in X: {missing_columns_X}")
        df_clean = df_dedup.dropna(subset=missing_columns_X).reset_index(drop=True)
        clean_row_count = len(df_clean)
        rows_dropped_missing = dedup_row_count - clean_row_count
        logging.info(f"Number of rows dropped due to missing values: {rows_dropped_missing}")
        logging.info(f"Number of rows after dropping missing values: {clean_row_count}")
        X = df_clean[features]
        y = df_clean['is_phishing']
        sample_ids = df_clean["sample_ids"]
        category = df_clean["category"]

    if y.isnull().any():
        missing_rows_y = y[y.isnull()].index.tolist()
        logging.warning(f"Target vector 'y' has missing values in rows: {missing_rows_y}")

    return X, y, sample_ids, category


In [18]:
def build_and_evaluate_models_per_category(X, y, sample_ids, category, path_prefix):
    numerical_features = [
        'message_length', 
        'url_count', 
        'total_messages', 
        'messages_repeat_by_user',
        'formatted_text_count'
    ]
    
    param_grid = {
        'n_estimators': [100, 200, 500],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced', 'balanced_subsample']
    }
    
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    
    results = {}
    best_params_dict = {}  # key: category, value: { cluster: list of best param dicts }
    agg_predictions = {}   # key: category, value: aggregated predictions dataframe

    unique_cats = sorted(category.dropna().unique())
    for cat in unique_cats:
        results[cat] = {
            'precision_neg': [],
            'recall_neg': [],
            'f1_neg': [],
            'precision_pos': [],
            'recall_pos': [],
            'f1_pos': [],
            'sensitivity': [],
            'specificity': [],
            'gmean': []
        }
        best_params_dict[cat] = {0: [], 1: [], 2: []}
        agg_predictions[cat] = []  # will store dataframes with predictions
    
    fold_idx = 0
    for outer_train_index, outer_test_index in outer_cv.split(X, y):
        fold_idx += 1
        logging.info(f"Outer Fold {fold_idx} starting...")
        X_outer_train = X.iloc[outer_train_index].reset_index(drop=True)
        y_outer_train = y.iloc[outer_train_index].reset_index(drop=True)
        sample_ids_outer_train = sample_ids.iloc[outer_train_index].reset_index(drop=True)
        category_outer_train = category.iloc[outer_train_index].reset_index(drop=True)
        
        X_outer_test = X.iloc[outer_test_index].reset_index(drop=True)
        y_outer_test = y.iloc[outer_test_index].reset_index(drop=True)
        sample_ids_outer_test = sample_ids.iloc[outer_test_index].reset_index(drop=True)
        category_outer_test = category.iloc[outer_test_index].reset_index(drop=True)
        
        for cat in unique_cats:
            # Select training and test samples for the given category.
            train_mask = (category_outer_train == cat)
            X_train_cat = X_outer_train[train_mask].reset_index(drop=True)
            y_train_cat = y_outer_train[train_mask].reset_index(drop=True)
            
            test_mask = (category_outer_test == cat)
            X_test_cat = X_outer_test[test_mask].reset_index(drop=True)
            y_test_cat = y_outer_test[test_mask].reset_index(drop=True)
            sample_ids_test_cat = sample_ids_outer_test[test_mask].reset_index(drop=True)
            
            if len(X_train_cat) < 10:
                logging.warning(f"Not enough training samples for category {cat} in outer fold {fold_idx}. Skipping.")
                continue
            
            # Scale numerical features.
            scaler = MaxAbsScaler()
            scaler.fit(X_train_cat[numerical_features])
            X_train_cat_scaled = X_train_cat.copy()
            X_test_cat_scaled = X_test_cat.copy()
            X_train_cat_scaled[numerical_features] = scaler.transform(X_train_cat[numerical_features])
            if not X_test_cat_scaled.empty:
                X_test_cat_scaled[numerical_features] = scaler.transform(X_test_cat[numerical_features])
            
            kmeans = KMeans(n_clusters=3, random_state=0)
            train_clusters = kmeans.fit_predict(X_train_cat_scaled)
            
            cluster_classifiers = {}
            cluster_majority_labels = {}
            
            # For each cluster in the training data, perform inner CV and train a classifier.
            for clust in [0, 1, 2]:
                cluster_mask = (train_clusters == clust)
                X_train_cluster = X_train_cat_scaled[cluster_mask]
                y_train_cluster = y_train_cat[cluster_mask]
                
                if len(X_train_cluster) < 10:
                    logging.warning(
                        f"Not enough training samples for category {cat}, cluster {clust} in outer fold {fold_idx}. "
                        "Skipping this cluster."
                    )
                    # If the cluster still has some data, use that cluster's majority label
                    if len(y_train_cluster) > 0:
                        cluster_majority_labels[clust] = y_train_cluster.value_counts().idxmax()
                    else:
                        # If the cluster is truly empty, fallback to the category majority
                        cluster_majority_labels[clust] = 0
                    # We don't train a classifier for this cluster
                    continue
                
                inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
                model_to_tune = RandomForestClassifier(random_state=0)
                grid_search = GridSearchCV(estimator=model_to_tune, param_grid=param_grid,
                                           cv=inner_cv, n_jobs=-1, scoring='f1_macro')
                grid_search.fit(X_train_cluster, y_train_cluster)
                best_params = grid_search.best_params_
                best_params_dict[cat][clust].append(best_params)
                
                model_cluster = RandomForestClassifier(**best_params, random_state=0)
                model_cluster.fit(X_train_cluster, y_train_cluster)
                cluster_classifiers[clust] = model_cluster
            
            # On the test set for the category, assign clusters using the trained k-means model.
            if not X_test_cat_scaled.empty:
                test_clusters = kmeans.predict(X_test_cat_scaled)
                y_pred_cat = []
                used_clusters = []
                for i, cl in enumerate(test_clusters):
                    if cl in cluster_classifiers:
                        pred = cluster_classifiers[cl].predict(X_test_cat_scaled.iloc[[i]])[0]
                    else:
                        pred = cluster_majority_labels[cl]
                        logging.warning(f"No classifier for category {cat} cluster {cl} in fold {fold_idx}; "f"defaulting to majority label = {pred}.")
                    y_pred_cat.append(pred)
                    used_clusters.append(cl)
                
                # Compute confusion matrix and sensitivity/specificity.
                cm = confusion_matrix(y_test_cat, y_pred_cat)
                TN = cm[0, 0] if cm.shape[0] > 1 and cm.shape[1] > 1 else 0
                FP = cm[0, 1] if cm.shape[1] > 1 else 0
                FN = cm[1, 0] if cm.shape[0] > 1 else 0
                TP = cm[1, 1] if cm.shape[0] > 1 and cm.shape[1] > 1 else 0
                sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
                specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
                gmean = np.sqrt(sensitivity * specificity)
                
                # Calculate metrics for both classes.
                precisions, recalls, f1_scores, _ = precision_recall_fscore_support(
                    y_test_cat, y_pred_cat, labels=[0, 1], zero_division=0)
                
                results[cat]['precision_neg'].append(precisions[0])
                results[cat]['recall_neg'].append(recalls[0])
                results[cat]['f1_neg'].append(f1_scores[0])
                results[cat]['precision_pos'].append(precisions[1])
                results[cat]['recall_pos'].append(recalls[1])
                results[cat]['f1_pos'].append(f1_scores[1])
                results[cat]['sensitivity'].append(sensitivity)
                results[cat]['specificity'].append(specificity)
                results[cat]['gmean'].append(gmean)
                
                # Save predictions for this fold and category.
                fold_data = X_test_cat.copy()
                fold_data['sample_ids'] = sample_ids_test_cat
                fold_data['actual'] = y_test_cat
                fold_data['predicted'] = y_pred_cat
                fold_data['predicted_cluster'] = used_clusters
                fold_data['category'] = cat
                fold_file = os.path.join(path_prefix, f"{cat}_fold{fold_idx}_predictions.csv")
                fold_data.to_csv(fold_file, index=False)
                
                # Aggregate predictions to later save a complete file.
                agg_predictions[cat].append(fold_data)
                
                print(f"Outer Fold {fold_idx} - Category {cat}:")
                print(pd.DataFrame(cm,
                                   index=['Non-Malicious (Negative)', 'Malicious (Positive)'],
                                   columns=['Predicted Non-Malicious', 'Predicted Malicious']))
                print(f"   Negative class - Precision: {precisions[0]:.3f}, Recall: {recalls[0]:.3f}, F1-score: {f1_scores[0]:.3f}")
                print(f"   Positive class - Precision: {precisions[1]:.3f}, Recall: {recalls[1]:.3f}, F1-score: {f1_scores[1]:.3f}")
                print(f"   Sensitivity (TPR): {sensitivity:.3f}")
                print(f"   Specificity (TNR): {specificity:.3f}")
                print(f"   G-mean: {gmean:.3f}\n")
            else:
                logging.info(f"No test samples for category {cat} in outer fold {fold_idx}.")
    
    # Print aggregated metrics per category (across outer folds).
    print("\nAggregated Metrics per Category (across outer folds):")
    for cat in unique_cats:
        if results[cat]['precision_neg']:
            prec_neg = np.array(results[cat]['precision_neg'])
            rec_neg = np.array(results[cat]['recall_neg'])
            f1_neg = np.array(results[cat]['f1_neg'])
            prec_pos = np.array(results[cat]['precision_pos'])
            rec_pos = np.array(results[cat]['recall_pos'])
            f1_pos = np.array(results[cat]['f1_pos'])
            sens_list = np.array(results[cat]['sensitivity'])
            spec_list = np.array(results[cat]['specificity'])
            gmean_list = np.array(results[cat]['gmean'])
            
            print(f"\nCategory: {cat}")
            print(" Negative class:")
            print(f"   Precision: {prec_neg.mean():.3f} ± {prec_neg.std():.3f}")
            print(f"   Recall:    {rec_neg.mean():.3f} ± {rec_neg.std():.3f}")
            print(f"   F1-score:  {f1_neg.mean():.3f} ± {f1_neg.std():.3f}")
            print(" Positive class:")
            print(f"   Precision: {prec_pos.mean():.3f} ± {prec_pos.std():.3f}")
            print(f"   Recall:    {rec_pos.mean():.3f} ± {rec_pos.std():.3f}")
            print(f"   F1-score:  {f1_pos.mean():.3f} ± {f1_pos.std():.3f}")
            print(f" Sensitivity: {sens_list.mean():.3f} ± {sens_list.std():.3f}")
            print(f" Specificity: {spec_list.mean():.3f} ± {spec_list.std():.3f}")
            print(f" G-mean:      {gmean_list.mean():.3f} ± {gmean_list.std():.3f}")
        else:
            print(f"\nCategory: {cat} - No evaluation metrics available.")
    
    # Aggregate best parameters per category and per cluster (using mode across outer folds).
    aggregated_best_params = {}
    param_types = {
        'n_estimators': lambda x: int(x) if pd.notnull(x) else None,
        'max_depth': lambda x: int(x) if pd.notnull(x) and x is not None else None,
        'min_samples_split': lambda x: int(x) if pd.notnull(x) else None,
        'class_weight': lambda x: x if pd.notnull(x) else None
    }
    for cat, clusters_params in best_params_dict.items():
        aggregated_best_params[cat] = {}
        for clust, params_list in clusters_params.items():
            if params_list:
                best_params_df = pd.DataFrame(params_list)
                aggregated_params = best_params_df.mode().iloc[0].to_dict()
                for param, conv in param_types.items():
                    if param in aggregated_params:
                        aggregated_params[param] = conv(aggregated_params[param])
                aggregated_best_params[cat][clust] = aggregated_params
                logging.info(f"Aggregated best parameters for category {cat}, cluster {clust}: {aggregated_params}")
            else:
                logging.info(f"No best parameters collected for category {cat}, cluster {clust}.")
    
    # Save aggregated best parameters to file.
    best_params_rows = []
    for cat, clust_dict in aggregated_best_params.items():
        for clust, params in clust_dict.items():
            row = {'category': cat, 'cluster': clust}
            row.update(params)
            best_params_rows.append(row)
    best_params_df_final = pd.DataFrame(best_params_rows)
    best_params_file = os.path.join(path_prefix, "aggregated_best_params_per_category_and_cluster.csv")
    best_params_df_final.to_csv(best_params_file, index=False)
    
    # Save aggregated predictions per category.
    for cat in unique_cats:
        if agg_predictions[cat]:
            all_preds = pd.concat(agg_predictions[cat], axis=0).reset_index(drop=True)
            agg_pred_file = os.path.join(path_prefix, f"{cat}_aggregated_predictions.csv")
            all_preds.to_csv(agg_pred_file, index=False)
    
    return aggregated_best_params


In [19]:
def train_final_models_per_category(X, y, sample_ids, category, path_prefix, aggregated_best_params):
   numerical_features = [
        'message_length', 
        'url_count', 
        'total_messages', 
        'messages_repeat_by_user',
        'formatted_text_count'
    ]
   for cat, clust_params in aggregated_best_params.items():
        mask = (category == cat)
        X_cat = X[mask].reset_index(drop=True)
        y_cat = y[mask].reset_index(drop=True)
        sample_ids_cat = sample_ids[mask].reset_index(drop=True)
        
        if len(X_cat) == 0:
            logging.info(f"No samples for category {cat}. Skipping final model training.")
            continue
        
        scaler = MaxAbsScaler()
        scaler.fit(X_cat[numerical_features])
        X_cat_scaled = X_cat.copy()
        X_cat_scaled[numerical_features] = scaler.transform(X_cat[numerical_features])
        
        # Fit final k-means on the entire category.
        kmeans_final = KMeans(n_clusters=3, random_state=0)
        final_clusters = kmeans_final.fit_predict(X_cat_scaled)
        
        # Train classifier for each cluster.
        for clust in [0, 1, 2]:
            cluster_mask = (final_clusters == clust)
            X_cat_cluster = X_cat_scaled[cluster_mask]
            y_cat_cluster = y_cat[cluster_mask]
            
            if len(X_cat_cluster) < 10:
                logging.warning(f"Not enough samples for final training for category {cat}, cluster {clust}. Skipping.")
                continue
            
            best_params = clust_params.get(clust)
            if best_params is None:
                logging.warning(f"No aggregated best parameters for category {cat}, cluster {clust}. Skipping.")
                continue
            
            model_final = RandomForestClassifier(**best_params, random_state=0)
            start_time = time.time()
            model_final.fit(X_cat_cluster, y_cat_cluster)
            duration = time.time() - start_time
            hours, rem = divmod(duration, 3600)
            minutes, seconds = divmod(rem, 60)
            logging.info(f"Final model for category {cat}, cluster {clust} trained in {int(hours)}h {int(minutes)}m {seconds:.2f}s")
            
            # Save the final classifier.
            model_file = os.path.join(path_prefix, f"{cat}_final_classifier_cluster_{clust}.pkl")
            joblib.dump(model_final, model_file)
            
        # Save the final k-means clustering model and scaler for the category.
        kmeans_file = os.path.join(path_prefix, f"{cat}_final_kmeans.pkl")
        scaler_file = os.path.join(path_prefix, f"{cat}_scaler.pkl")
        joblib.dump(kmeans_final, kmeans_file)
        joblib.dump(scaler, scaler_file)
        
        # Save final models' parameters (for record).
        params_file = os.path.join(path_prefix, f"{cat}_final_model_params.txt")
        with open(params_file, 'w') as f:
            f.write("Aggregated best parameters per cluster:\n")
            for clust, params in clust_params.items():
                f.write(f"Cluster {clust}: {params}\n")
        logging.info(f"Final k-means, classifiers and scaler for category {cat} saved.")


In [20]:
def models_per_category(df, path_prefix):
    selected_features = [
        'is_bot', 'message_length', 'has_media', 'unique_users_per_group_message',
        'is_reply', 'url_count', 'total_messages', 'normalized_days_until_first_post',
        'language_match', 'messages_repeat_by_user', 'username_matches_language', 'formatted_text_count'
    ]
    X, y, sample_ids, category = prepare_features(df, selected_features)
    
    logging.info(f"Unique target labels: {np.unique(y, return_counts=True)}")
    logging.info(f"Unique categories: {np.unique(category)}")
    
    os.makedirs(path_prefix, exist_ok=True)
    
    aggregated_best_params = build_and_evaluate_models_per_category(X, y, sample_ids, category, path_prefix)
    
    train_final_models_per_category(X, y, sample_ids, category, path_prefix, aggregated_best_params)


In [21]:
def main():
    save_path = '3cluster-results/'
    input_path = '../../../data/'

    os.makedirs(save_path, exist_ok=True)
    df = load_data(input_path)
    df = preprocess_data(df)

    models_per_category(df, save_path)


In [22]:
if __name__ == '__main__':
    main()

INFO:root:Loading data...
INFO:root:Preprocessing data...
INFO:root:Preparing features and target variables...
INFO:root:Number of duplicate rows dropped: 7315
INFO:root:Number of rows dropped due to missing values: 1017
INFO:root:Number of rows after dropping missing values: 49877
INFO:root:Unique target labels: (array([False,  True]), array([48880,   997]))
INFO:root:Unique categories: ['Crypto' 'Darknet' 'Games']
INFO:root:Outer Fold 1 starting...


Outer Fold 1 - Category Crypto:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     5714                    6
Malicious (Positive)                           23                   20
   Negative class - Precision: 0.996, Recall: 0.999, F1-score: 0.997
   Positive class - Precision: 0.769, Recall: 0.465, F1-score: 0.580
   Sensitivity (TPR): 0.465
   Specificity (TNR): 0.999
   G-mean: 0.682


Outer Fold 1 - Category Darknet:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                      539                   22
Malicious (Positive)                            5                    6
   Negative class - Precision: 0.991, Recall: 0.961, F1-score: 0.976
   Positive class - Precision: 0.214, Recall: 0.545, F1-score: 0.308
   Sensitivity (TPR): 0.545
   Specificity (TNR): 0.961
   G-mean: 0.724


INFO:root:Outer Fold 2 starting...


Outer Fold 1 - Category Games:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     3473                   22
Malicious (Positive)                           29                  117
   Negative class - Precision: 0.992, Recall: 0.994, F1-score: 0.993
   Positive class - Precision: 0.842, Recall: 0.801, F1-score: 0.821
   Sensitivity (TPR): 0.801
   Specificity (TNR): 0.994
   G-mean: 0.892
Outer Fold 2 - Category Crypto:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     5798                    9
Malicious (Positive)                           26                   13
   Negative class - Precision: 0.996, Recall: 0.998, F1-score: 0.997
   Positive class - Precision: 0.591, Recall: 0.333, F1-score: 0.426
   Sensitivity (TPR): 0.333
   Specificity (TNR): 0.998
   G-mean: 0.577


...

/Users/minaerfan/researchProjects/pythonProject/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Outer Fold 5 - Category Darknet:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                      542                   18
Malicious (Positive)                            6                   13
   Negative class - Precision: 0.989, Recall: 0.968, F1-score: 0.978
   Positive class - Precision: 0.419, Recall: 0.684, F1-score: 0.520
   Sensitivity (TPR): 0.684
   Specificity (TNR): 0.968
   G-mean: 0.814


INFO:root:Aggregated best parameters for category Crypto, cluster 0: {'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 200}
INFO:root:Aggregated best parameters for category Crypto, cluster 1: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
INFO:root:Aggregated best parameters for category Crypto, cluster 2: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}
INFO:root:Aggregated best parameters for category Darknet, cluster 0: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
INFO:root:No best parameters collected for category Darknet, cluster 1.
INFO:root:Aggregated best parameters for category Darknet, cluster 2: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
INFO:root:Aggregated best parameters for category Games, cluster 0: {'class_weight': 'balanced', 'max_depth': 

Outer Fold 5 - Category Games:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     3398                   24
Malicious (Positive)                           22                  114
   Negative class - Precision: 0.994, Recall: 0.993, F1-score: 0.993
   Positive class - Precision: 0.826, Recall: 0.838, F1-score: 0.832
   Sensitivity (TPR): 0.838
   Specificity (TNR): 0.993
   G-mean: 0.912


Aggregated Metrics per Category (across outer folds):

Category: Crypto
 Negative class:
   Precision: 0.996 ± 0.000
   Recall:    0.998 ± 0.001
   F1-score:  0.997 ± 0.000
 Positive class:
   Precision: 0.626 ± 0.098
   Recall:    0.386 ± 0.050
   F1-score:  0.476 ± 0.062
 Sensitivity: 0.386 ± 0.050
 Specificity: 0.998 ± 0.001
 G-mean:      0.620 ± 0.040

Category: Darknet
 Negative class:
   Precision: 0.985 ± 0.005
   Recall:    0.979 ± 0.013
   F1-score:  0.982 ± 0.006
 Positive class:
   Precision: 0.422 ± 0.171
   Recall:    0.450 

INFO:root:Final model for category Crypto, cluster 0 trained in 0h 0m 0.88s
INFO:root:Final model for category Crypto, cluster 1 trained in 0h 0m 0.05s
INFO:root:Final model for category Crypto, cluster 2 trained in 0h 0m 0.48s
INFO:root:Final k-means, classifiers and scaler for category Crypto saved.
INFO:root:Final model for category Darknet, cluster 0 trained in 0h 0m 0.24s
INFO:root:Final model for category Darknet, cluster 2 trained in 0h 0m 0.05s
INFO:root:Final k-means, classifiers and scaler for category Darknet saved.
INFO:root:Final model for category Games, cluster 0 trained in 0h 0m 0.20s
INFO:root:Final model for category Games, cluster 1 trained in 0h 0m 0.19s
INFO:root:Final model for category Games, cluster 2 trained in 0h 0m 0.16s
INFO:root:Final k-means, classifiers and scaler for category Games saved.
